## Day 5 
Final Data Fusion Pipeline

In [1]:
import pandas as pd
import numpy as np

In [2]:
def load_dataset(path):
    try:
        df = pd.read_csv(path, parse_dates=["timestamp"])
        print("Dataset loaded")
        return df
    except FileNotFoundError:
        print("File not found")
        return None
    except Exception as e:
        print("Error:", e)
        return None


def create_context_data(df):
    context_df = pd.DataFrame()
    context_df["timestamp"] = df["timestamp"]
    context_df["shift"] = pd.cut(df["timestamp"].dt.hour, bins=[-1, 7, 15, 23], labels=["Night", "Morning", "Evening"])
    np.random.seed(42)
    context_df["production_load"] = np.random.uniform(0.6, 1.0, len(df))
    context_df["power_quality_index"] = np.random.uniform(0.85, 1.0, len(df))
    context_df["maintenance_window"] = (df["Tool wear [min]"] // 120).astype(int)
    context_df["operator_experience"] = np.random.choice([1, 2, 3, 4, 5], size=len(df))
    return context_df


def data_fusion(df, context_df):
    merged_df = pd.merge(df, context_df, on="timestamp", how="left")
    merged_df["wear_load_index"] = (merged_df["Tool wear [min]"] * merged_df["production_load"])
    merged_df["temperature_gap"] = (merged_df["Process temperature [K]"] - merged_df["Air temperature [K]"])
    merged_df["power_stress_score"] = (merged_df["Torque [Nm]"] * (1 - merged_df["power_quality_index"]))
    return merged_df


def validate_data(df):
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicates:", df.duplicated().sum())
    print("Missing Values:", df.isnull().sum().sum())
    if df.isnull().sum().sum() == 0:
        print("Validation Passed")
    else:
        print("Validation Failed")
    if df["timestamp"].isnull().sum() > 0:
        raise ValueError("Null timestamps detected")
    if df["timestamp"].duplicated().sum() > 0:
        raise ValueError("Duplicate timestamps detected")
    print("Timestamp validation passed")

    if df["production_load"].min() < 0 or df["production_load"].max() > 1:
        raise ValueError("production_load out of range")
    if df["power_quality_index"].min() < 0 or df["power_quality_index"].max() > 1:
        raise ValueError("power_quality_index out of range")
    print("Context validation passed")


def calculate_quality_score(df):
    missing_score = (1 - (df.isnull().sum().sum() / (len(df) * len(df.columns)))) * 100
    duplicate_score = (1 - (df.duplicated().sum() / len(df))) * 100
    quality_score = (missing_score * 0.5 + duplicate_score * 0.5)
    return round(quality_score, 2)

In [3]:
df = load_dataset("predictive_maintainence_timestamp.csv")

if df is None:
    raise SystemExit("Pipeline stopped: dataset failed to load")

context_df = create_context_data(df)
merged_df = data_fusion(df, context_df)
validate_data(merged_df)

score = calculate_quality_score(merged_df)
print("Fusion Quality Score:", score)

In [4]:
bad_df = merged_df.copy()
bad_df.loc[0, "timestamp"] = None
try:
    validate_data(bad_df)
except ValueError as e:
    print("Caught expected error:", e)

In [5]:
merged_df.to_csv("fusion_pipeline_output_consolidated.csv", index=False)
print("Pipeline completed")